# BioPred Phase 0 -- Activity Evidence Cleaning

## Purpose

This notebook loads the raw BioPred GABA-A activity evidence table produced in `02_activity_evidence_ingestion.ipynb` and evaluates whether the extracted data is suitable for downstream modeling.

The goal is not to train models or generate molecular features. The goal is to apply basic usability filters, examine endpoint/value quality, define preliminary cleaning policies, and assess whether the dataset has enough clean molecule-level evidence to justify the next BioPred phase.

## Phase context

This notebook begins the transition from **Phase 0: discovery and data viability assessment** into **Phase 1: curated evidence dataset construction**.

Notebook 02 preserved raw ChEMBL activity evidence at `activity_id` grain. This notebook begins reducing that raw evidence into a cleaner, auditable dataset while preserving endpoint, target, assay, molecule, and source metadata.

## Notebook Objectives

1. Load the provisional ChEMBL activity evidence Parquet artifact from Notebook 2.
2. Audit row counts, unique molecules, endpoints, units, relations, and missing values before filtering.
3. Apply basic usability filters for molecule structure, endpoint type, concentration units, numeric values, and relation policy.
4. Compute standardized pX potency values for valid concentration-based activity records.
5. Explore candidate activity-label thresholds and resulting class prevalence.
6. Audit endpoint and target composition after cleaning.
7. Identify duplicate or repeated molecule-endpoint-target measurements for downstream aggregation policy.
8. Save a cleaned activity evidence artifact for feature engineering, aggregation, and split planning.

## Design principle

Clean conservatively and audit each reduction step. BioPred should preserve enough metadata to explain which records were removed, which records remained, and why the resulting dataset is or is not viable for ranking-model development.

In [2]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import numpy as np

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

load_dotenv(paths.PROJECT_ROOT / ".env", override=True)

# create the database engine
db_url = URL.create(
    drivername = os.getenv("BIOPRED_DB_DIALECT"),
    username=os.getenv("BIOPRED_DB_USER"),
    password=os.getenv("BIOPRED_DB_PASSWORD"),
    host=os.getenv("BIOPRED_DB_HOST"),
    port=int(os.getenv("BIOPRED_DB_PORT")),
    database=os.getenv("BIOPRED_DB_NAME")
)

engine = create_engine(db_url, pool_pre_ping=True)
schema = os.getenv("BIOPRED_DB_SCHEMA", "public")

/home/ryanm/projects/BioPred


In [3]:
# load our activity_evidence_raw artifact we created from the last notebook
activity_evidence_path = paths.INTERIM_DATA_DIR / "gabaa_activity_evidence_raw.parquet"

activity_evidence_raw = pd.read_parquet(activity_evidence_path)



In [4]:
# Verify shape
activity_evidence_raw.shape

# Looking for (5579, 21)

(5579, 21)

### **Section 2: Baseline Activity Audit**

This section audits the provisional activity evidence artifact before applying usability filters. The goal is to understand row counts, molecule coverage, endpoint composition, units, relation operators, missingness, and key identifier fields.

In [5]:
# We already confirmed shape, but let's be a bit more explicit about it as well as rename the key variable for ease of use here.
df = activity_evidence_raw

n_unique_molecules = df["molecule_chembl_id"].nunique()
n_unique_smiles = df["canonical_smiles"].nunique()

n_unique_molecules, n_unique_smiles

(1960, 1960)

In [6]:
# Now let's make a table that shows dtype, missing count, missing %, unique count, and example(just first 5) of non-null values.
col_audit = pd.DataFrame({
    "dtype" : df.dtypes.astype(str),
    "missing_count" : df.isnull().sum(),
    "missing_percent" : df.isnull().mean() * 100,
    "unique_count" : df.nunique(),
    "example_non_null" : df.apply(lambda x: x.dropna().unique()[0] if x.dropna().nunique() > 0 else None)
})

example_values = df.apply(
    lambda s : s.dropna().unique()[:5]
)

col_audit_table = (
    col_audit
    .assign(example_values=example_values)
    .sort_values(["missing_percent", "unique_count"], ascending=[False, True])
)

col_audit_table

,dtype,missing_count,missing_percent,unique_count,example_non_null,example_values
max_phase,float64,5243,93.977415,5,4.0,"[4.0, 2.0, 3.0, 1.0, -1.0]"
pchembl_value,float64,1063,19.053594,547,4.43,"[4.43, 4.41, 4.77, 5.08, 5.7]"
standard_value,float64,152,2.724503,1515,36800.0,"[36800.0, 39300.0, 16900.0, 8400.0, 2000.0]"
standard_relation,object,148,2.652805,5,=,"[=, >, <, ~, >=]"
molecule_type,object,140,2.509410,3,Small molecule,"[Small molecule, Unknown, Protein]"
standard_units,object,103,1.846209,3,nM,"[nM, microA, %]"
organism,object,0,0.000000,1,Homo sapiens,[Homo sapiens]
target_type,object,0,0.000000,2,PROTEIN COMPLEX,"[PROTEIN COMPLEX, PROTEIN COMPLEX GROUP]"
standard_type,object,0,0.000000,3,EC50,"[EC50, IC50, Ki]"
assay_type,object,0,0.000000,4,B,"[B, F, T, A]"


**Key Observations**

`max_phase` is very sparse, showing 93.98% missingness.  This is expected though, since most of these molecules are not approved or clinically advanced drugs.  This column is useful downstream for later rediscovery and annotation but won't be for now and in the modeling phase.

`pchembl_value` also has notable missingness, at 19%.  We need to investigate further, to see if the missingness here is at random before blindly discarding this feature.

`standard_value` has good coverage. With missingness only at 2.72% we can feel confident moving forward computing our own pX values.

`standard_units` contains mixed measurement types.  This is a really interesting read.  We are really only interested in **nM** here for concentration-based potency.  MicroA and % are not relevant for us in this project.  We will need to consider this and use a filter to acquire just molecules with **nM** associated with them.

`standard_type` has three endpoints, as expected.  They are all related but not necessarily the same.  For the first pass into our ranking model we will consider converting them all into concentration endpoints of pX values.

`standard_relation` includes non-exact values, like >, <, >=, etc.  For computing exact pX values, the cleanest measure is only including rows in this column that have either '=' or '=='.

`molecule_type` includes molecule types that aren't just **small_molecule**.  This is important, as BioPred is designed as a small-molecule ligand ranking project.  So we will need to keep it clean and filter for just **small_molecule** at this time.


In [7]:
 # Now let's look at some endpoint/unit/relation compositions
composition_cols = [
    "standard_type",
    "standard_units",
    "standard_relation",
    "molecule_type",
    "assay_type",
    "target_type",
    "confidence_score"
]

for col in composition_cols:
    display(
        df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="n_records")
        .assign(pct=lambda x: (x["n_records"] / len(df)).round(4))
    )


,standard_type,n_records,pct
0,Ki,4321,0.7745
1,EC50,677,0.1213
2,IC50,581,0.1041


,standard_units,n_records,pct
0,nM,5473,0.9810
1,None,103,0.0185
2,%,2,0.0004
3,microA,1,0.0002


,standard_relation,n_records,pct
0,=,4703,0.8430
1,>,724,0.1298
2,None,148,0.0265
3,<,2,0.0004
4,~,1,0.0002
5,>=,1,0.0002


,molecule_type,n_records,pct
0,Small molecule,5279,0.9462
1,Unknown,159,0.0285
2,None,140,0.0251
3,Protein,1,0.0002


,assay_type,n_records,pct
0,B,5286,0.9475
1,F,271,0.0486
2,A,19,0.0034
3,T,3,0.0005


,target_type,n_records,pct
0,PROTEIN COMPLEX,4820,0.864
1,PROTEIN COMPLEX GROUP,759,0.136


,confidence_score,n_records,pct
0,7,3002,0.5381
1,6,1818,0.3259
2,4,582,0.1043
3,5,177,0.0317


**Baseline Evidence Composition**

Before applying usability filters, we looked at the distribution of key activity-evidence fields above.  This audit will help determine which records are suitable for standardized potency modeling.  The provisional dataset we have been using is dominated by **Ki** measurements, uses **nM** units for nearly all records, and primarily contains exact (**=**) standard relations.  These patterns support a conservative first-pass cleaning policy focused on **small-molecule**, concentration-based records with exact nanomolar activity values.

Rows with non-concentration units, missing units, non-small-molecule records, or approximate relations are still retained at this audit stage but will be evaluated during the filter step before being excluded from the cleaned evidence artifact.

Now we will make a Filter Attrition Audit, which will show the results fo what happens to our data when we perform each of our proposed cleaning rules.  How much evidence do we lose?  Which rule has the most effect?  After executing all of our proposed rules do we still have enough evidence for downstream modeling?

In [8]:
# first make a list of all of our proposed filtering steps
filter_steps = [
    ("raw", pd.Series(True, index = df.index)),
    ("non_null_smiles", df["canonical_smiles"].notna()),
    ("small_molecule", df["molecule_type"].eq("Small molecule")),
    ("concentration_endpoint", df["standard_type"].isin(["Ki", "IC50", "EC50"])),
    ("nM_units", df["standard_units"].eq("nM")),
    ("exact_relation", df["standard_relation"].eq("=")),
    ("non_null_standard_value", df["standard_value"].notna()),
    ("positive_standard_value", df["standard_value"].gt(0)),
]

# make a loop now with a mask that goes through all of our steps
current_mask = pd.Series(True, index=df.index)

baseline_records = len(df)
baseline_mols = df["molecule_chembl_id"].nunique()

previous_records = baseline_records
previous_mols = baseline_mols

rows = []

for step_name, step_mask in filter_steps:
    current_mask = current_mask & step_mask

    n_records = current_mask.sum()
    n_unique_mols = df.loc[current_mask, "molecule_chembl_id"].nunique()

    rows.append({
        "step" : step_name,
        "n_records" : n_records,
        "n_unique_molecules" : n_unique_mols,
        "records_removed_from_prev_step" : previous_records - n_records,
        "molecules_removed_from_prev_step" : previous_mols - n_unique_mols,
        "pct_records_rem" : n_records / baseline_records,
        "pct_mols_rem" : n_unique_mols / baseline_mols
    })

    previous_records = n_records
    previous_mols = n_unique_mols

attrition = pd.DataFrame(rows)

attrition


,step,n_records,n_unique_molecules,records_removed_from_prev_step,molecules_removed_from_prev_step,pct_records_rem,pct_mols_rem
0,raw,5579,1960,0,0,1.000000,1.000000
1,non_null_smiles,5579,1960,0,0,1.000000,1.000000
2,small_molecule,5279,1768,300,192,0.946227,0.902041
3,concentration_endpoint,5279,1768,0,0,0.946227,0.902041
4,nM_units,5187,1737,92,31,0.929737,0.886224
5,exact_relation,4439,1546,748,191,0.795662,0.788776
6,non_null_standard_value,4435,1546,4,0,0.794945,0.788776
7,positive_standard_value,4435,1546,0,0,0.794945,0.788776


This table is really useful, it helps us document what and why we should filter and what their effects would be on our provisional dataset.  Rather than just go right into filtering not knowing, we can see several things here:

- The dataset is not only mainly being reduced by endpoint type or missing structures, the main cleaning decision is relation policy (exact_relation).
- Even though we stand to lose a little over 20% of our existing dataset, this is still rather conservative with the rules we have in place here.

We will go ahead and use these filters to create our filtered dataframe.

In [9]:
# create another mask to use for our rules
rules_mask = (
    df["canonical_smiles"].notna()
    & df['molecule_type'].eq('Small molecule')
    & df['standard_type'].isin(['Ki', 'IC50', 'EC50'])
    & df['standard_units'].eq('nM')
    & df['standard_relation'].eq('=')
    & df['standard_value'].notna()
    & df['standard_value'].gt(0)
)

# apply mask to our df
df_clean = df.loc[rules_mask].copy()

# check shape of new df as well as expected unique mols, verifying with our attrition table above
print(df_clean.shape)
print(df_clean["molecule_chembl_id"].nunique()) # Expecting 1546

(4435, 21)
1546


Before we move on to the next section, there is one erroneous task we need to do for general purpose cleaning.

In [10]:
# reassign/change the name of one main col, as it isn't name-friendly
df_clean = df_clean.rename(columns = {"assay_cheml_id" : "assay_chembl_id"})

# Quick verification check
"assay_chembl_id" in df_clean.columns, "assay_cheml_id" in df_clean.columns #Expecting (True,False)

(True, False)

### **Section 4: Compute Standardized pX Potency Values**

The cleaned evidence dataset contains concentration-based `Ki`, `IC50`, and `EC50` measurements reported in nanomolar units. We convert these raw concentrations into standardized pX values using `pX = -log10(concentration in M)`, where higher pX values indicate stronger potency or affinity.

This transformation puts all retained activity evidence on a common directionally consistent scale for threshold exploration, aggregation analysis, and downstream modeling.

In [11]:
# Convert nM concentration values to molar concentration
df_clean["standard_value_molar"] = df_clean["standard_value"] * 1e-9

# make new col for our px_values
df_clean["px_value"] = -np.log10(df_clean['standard_value_molar'])

# We could have done this with:
# df_clean["px_value"] = 9 - np.log10(df_clean['standard_value'])
# But I wanted to show the longer transition to px_values

In [12]:
# let's validate our conversion with a sample
df_clean[
    ["standard_type", "standard_value", "standard_value_molar", "px_value", "pchembl_value"]
].head(10)

,standard_type,standard_value,standard_value_molar,px_value,pchembl_value
7,EC50,36800.0,0.000037,4.434152,4.43
8,EC50,39300.0,0.000039,4.405607,4.41
9,EC50,16900.0,0.000017,4.772113,4.77
10,EC50,8400.0,0.000008,5.075721,5.08
11,EC50,2000.0,0.000002,5.698970,5.70
12,EC50,3800.0,0.000004,5.420216,5.42
13,EC50,7700.0,0.000008,5.113509,5.11
14,EC50,18500.0,0.000019,4.732828,4.73
15,EC50,9300.0,0.000009,5.031517,5.03
16,EC50,9200.0,0.000009,5.036212,5.04


Our new px_value and pchembl_value are almost identical.  Let's make sure there aren't any huge differences by comparing them.

In [13]:
px_comparison = (
    df_clean
    .loc[
        df_clean["pchembl_value"].notna(),
        ["standard_type", "standard_value", "px_value", "pchembl_value"]
    ]
    .assign(px_diff=lambda x: x["px_value"] - x["pchembl_value"])
)

px_comparison.reindex(
    px_comparison["px_diff"].abs().sort_values(ascending=False).head(10).index
)

,standard_type,standard_value,px_value,pchembl_value,px_diff
5179,EC50,394.00,6.404504,6.41,-0.005496
1841,Ki,432.00,6.364516,6.37,-0.005484
5143,EC50,945.00,6.024568,6.03,-0.005432
5140,EC50,945.00,6.024568,6.03,-0.005432
5170,EC50,292.00,6.534617,6.54,-0.005383
5104,Ki,292.00,6.534617,6.54,-0.005383
3877,Ki,29.20,7.534617,7.54,-0.005383
895,Ki,4620.00,5.335358,5.33,0.005358
4174,Ki,2.02,8.694649,8.70,-0.005351
3238,Ki,20.20,7.694649,7.70,-0.005351


Good there is almost no meaningful difference between the two columns (seen by px_diff).  We will still use px_value as valid and use it going forward, as it was explicitly derived from our standard_value and our cleaned data, and that it is reproducible.  We will keep pchembl_value for references purposes only.

In [14]:
# summarize the potency distribution
df_clean["px_value"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99])

count    4435.000000
mean        7.172560
std         1.594321
min         0.079000
1%          1.000000
5%          4.356547
10%         5.232118
25%         6.414545
50%         7.387216
75%         8.251812
90%         9.000000
95%         9.301030
99%         9.814379
max        10.698970
Name: px_value, dtype: float64

In [15]:
# and look at endpoint-specific distribution as well
df_clean.groupby("standard_type")["px_value"].describe()

,count,mean,std,min,25%,50%,75%,max
standard_type,,,,,,,,
EC50,459.0,5.048611,1.529526,1.000000,4.301030,5.060481,6.032236,9.154902
IC50,460.0,5.993341,2.138997,0.079000,5.195609,6.322111,7.556791,10.000000
Ki,3516.0,7.604111,1.160515,2.309804,6.826087,7.638272,8.420216,10.698970


### Endpoint-Specific Potency Patterns

The earlier composition audit showed that the cleaned activity evidence dataset is dominated by `Ki` records. The pX distribution audit adds an important second finding: `Ki` records are not only more numerous, but also show substantially higher observed pX values than `IC50` and `EC50` records in this cleaned evidence set.

This matters because a single global activity-label threshold may affect endpoint groups differently. Since `Ki`, `IC50`, and `EC50` measure related but non-identical biological quantities, downstream threshold selection should be evaluated both overall and by endpoint before defining the final activity label.

### **Section 5: Explore Candidate Activity-Label Thresholds**

The cleaned evidence dataset now contains standardized `px_value` measurements for exact nanomolar `Ki`, `IC50`, and `EC50` records. In this section, we evaluate candidate pX cutoffs for defining active versus inactive evidence records.

This is a label-definition analysis, not model probability threshold tuning. The goal is to quantify overall and endpoint-specific class prevalence at several plausible potency thresholds before selecting a downstream activity-label policy.

In [19]:
# run an overall threshold summary for our pX values.  
candidate_thresholds = [5.0, 5.5, 6.0, 6.5, 7.0]

threshold_rows = []

for threshold in candidate_thresholds:
    active_mask = df_clean["px_value"].ge(threshold)

    threshold_rows.append({
        # create the calculations and data we want here
        "threshold" : threshold,
        "n_active" : active_mask.sum(),
        "pct_active" : round(active_mask.sum() / len(df_clean), 4),
        "n_inactive" : (~active_mask).sum(),
        "pct_inactive" : round((~active_mask).sum() / len(df_clean), 4)
    })

threshold_summary = pd.DataFrame(threshold_rows)

threshold_summary

,threshold,n_active,pct_active,n_inactive,pct_inactive
0,5.0,4060,0.9154,375,0.0846
1,5.5,3911,0.8818,524,0.1182
2,6.0,3653,0.8237,782,0.1763
3,6.5,3236,0.7297,1199,0.2703
4,7.0,2670,0.6020,1765,0.3980


**Key Interpretation:**

This dataset is very potency-rich.  Even at **pX >= 6.0**, about **82%** of evidence records are still active.  Also, when **pX** is set to **7.0** about **60%** remain active.  That means even **lowering** the **pX to >= 5.5** may still be too permissive if we want a useful binary label at the evidence-record level.  We need endpoint-specific prevalence before we make a decision.


In [21]:
# run another summary, this time for endpoint-specific threshold
# use candidate_thresholds var from previous in our for loop(s)

endpoint_threshold_rows = []

for endpoint, endpoint_df in df_clean.groupby("standard_type"):
    for threshold in candidate_thresholds:
        active_mask = endpoint_df["px_value"].ge(threshold)

        endpoint_threshold_rows.append({
            "standard_type" : endpoint,
            "threshold" : threshold,
            "n_records" : len(endpoint_df),
            "n_active" : active_mask.sum(),
            "pct_active" : round(active_mask.sum() / len(endpoint_df), 4),
            "n_inactive" : (~active_mask).sum(),
            "pct_inactive" : round((~active_mask).sum() / len(endpoint_df), 4)
        })

endpoint_threshold_summary = pd.DataFrame(endpoint_threshold_rows)

endpoint_threshold_summary

,standard_type,threshold,n_records,n_active,pct_active,n_inactive,pct_inactive
0,EC50,5.0,459,242,0.5272,217,0.4728
1,EC50,5.5,459,192,0.4183,267,0.5817
2,EC50,6.0,459,121,0.2636,338,0.7364
3,EC50,6.5,459,64,0.1394,395,0.8606
4,EC50,7.0,459,26,0.0566,433,0.9434
5,IC50,5.0,460,363,0.7891,97,0.2109
6,IC50,5.5,460,325,0.7065,135,0.2935
7,IC50,6.0,460,274,0.5957,186,0.4043
8,IC50,6.5,460,211,0.4587,249,0.5413
9,IC50,7.0,460,164,0.3565,296,0.6435


 ### Candidate Threshold Prevalence

Candidate pX thresholds were evaluated to understand how different activity-label cutoffs affect class prevalence in the cleaned evidence dataset. This analysis is focused on label definition, not model probability threshold tuning.

Overall, the cleaned evidence set is potency-rich: even at stricter cutoffs, a large fraction of records remain active. This means permissive thresholds such as `pX >= 5.0` or `pX >= 5.5` may create limited inactive contrast for downstream ranking evaluation.

In [22]:
# for readability turn that into a quick pivot table
endpoint_threshold_summary.pivot(
    index = "threshold",
    columns = "standard_type",
    values = "pct_active"
)

standard_type,EC50,IC50,Ki
threshold,,,
5.0,0.5272,0.7891,0.9827
5.5,0.4183,0.7065,0.9653
6.0,0.2636,0.5957,0.9266
6.5,0.1394,0.4587,0.8422
7.0,0.0566,0.3565,0.7053


### Endpoint-Specific Threshold Prevalence

Endpoint-specific threshold prevalence shows that a single global pX cutoff affects `Ki`, `IC50`, and `EC50` records very differently. `Ki` records remain highly active across all candidate thresholds, while `EC50` records become majority inactive at thresholds above `pX >= 5.0`.

This confirms that endpoint composition materially influences any binary activity label derived from the cleaned evidence set. As a result, candidate thresholds should be carried forward into molecule-level aggregation rather than finalized at the evidence-record stage.

We aren't ready to select the final label yet, we are just preserving candidate policies for molecule-level aggregation.

In [23]:
# create three binary columns for our candidate px values
candidate_label_thresholds = {
    "active_px_5_5" : 5.5,
    "active_px_6_0" : 6.0,
    "active_px_6_5" : 6.5,
}

for label_col, threshold in candidate_label_thresholds.items():
    df_clean[label_col] = df_clean["px_value"].ge(threshold).astype(int)

candidate_label_cols = list(candidate_label_thresholds)

df_clean[
    ["standard_type", "standard_value", "px_value", *candidate_label_cols]
].head(10)

df_clean[candidate_label_cols].mean().sort_values(ascending = False)

active_px_5_5    0.881849
active_px_6_0    0.823675
active_px_6_5    0.729651
dtype: float64

### Candidate Evidence-Level Label Columns

Three candidate binary activity labels were added to the cleaned evidence dataset using pX thresholds of 5.5, 6.0, and 6.5. These columns are evidence-level labels only; they are not yet final molecule-level modeling labels.

The active prevalence remains high across all three candidates, confirming that the cleaned GABA-A evidence set is potency-rich. These candidate labels will be retained for downstream repeated-measurement and aggregation analysis before selecting the final modeling label.

### **Section 6: Audit Endpoint and Target Composition After Cleaning**

After applying usability filters and computing standardized pX values, we audit the cleaned evidence set by endpoint and target composition. This step checks whether the retained activity evidence is broadly distributed across GABA-A targets or concentrated in a small number of receptor complexes.

This matters because downstream models may reflect the dominant targets and endpoint types present in the cleaned dataset. Auditing target-level record counts, molecule counts, potency distributions, and candidate-label prevalence helps identify possible target-composition bias before aggregation and feature engineering.

In [31]:
# produce a target summary, focusing and using cols target_chembl_id, target_pref_name and target_type
target_summary = (
    df_clean
    .groupby(["target_chembl_id", "target_pref_name", "target_type"])
    .agg(
        n_records = ("activity_id", "count"),
        n_unique_molecules = ("molecule_chembl_id", "nunique"),
        mean_px = ("px_value", "mean"),
        median_px = ("px_value", "median"),
        pct_active_px_5_5 = ("active_px_5_5", "mean"),
        pct_active_px_6_0 = ("active_px_6_0", "mean"),
        pct_active_px_6_5 = ("active_px_6_5", "mean"),
    )
    .reset_index()
    .sort_values("n_records", ascending = False)
)

target_summary

,target_chembl_id,target_pref_name,target_type,n_records,n_unique_molecules,mean_px,median_px,pct_active_px_5_5,pct_active_px_6_0,pct_active_px_6_5
3,CHEMBL2094121,GABA-A receptor; alpha-1/beta-3/gamma-2,PROTEIN COMPLEX,801,609,7.534227,7.573489,0.943820,0.923845,0.845194
4,CHEMBL2094122,GABA-A receptor; alpha-5/beta-3/gamma-2,PROTEIN COMPLEX,790,631,8.004009,8.251812,0.973418,0.948101,0.874684
2,CHEMBL2094120,GABA-A receptor; alpha-3/beta-3/gamma-2,PROTEIN COMPLEX,774,626,7.637236,7.677781,0.968992,0.941860,0.872093
5,CHEMBL2094130,GABA-A receptor; alpha-2/beta-3/gamma-2,PROTEIN COMPLEX,657,510,7.389808,7.468521,0.961948,0.923896,0.843227
1,CHEMBL2093872,GABA-A receptor; anion channel,PROTEIN COMPLEX GROUP,559,455,6.216162,6.621602,0.740608,0.654741,0.518784
6,CHEMBL2095172,GABA-A receptor; alpha-1/beta-2/gamma-2,PROTEIN COMPLEX,329,223,5.760863,5.849999,0.604863,0.468085,0.349544
7,CHEMBL2095190,GABA-A receptor; alpha-6/beta-3/gamma-2,PROTEIN COMPLEX,183,119,6.552706,6.744727,0.819672,0.693989,0.584699
13,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,PROTEIN COMPLEX,98,86,6.272040,6.083094,0.704082,0.530612,0.326531
9,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,PROTEIN COMPLEX,62,48,7.005439,7.187416,0.806452,0.741935,0.629032
10,CHEMBL2111366,GABA A receptor alpha-4/beta-3/gamma-2,PROTEIN COMPLEX,57,53,5.799658,6.000000,0.771930,0.508772,0.333333


### Target Composition After Cleaning

The cleaned evidence set is concentrated in a small number of GABA-A receptor complex annotations. The highest-volume targets are primarily alpha/beta-3/gamma-2 receptor complexes, including alpha-1, alpha-2, alpha-3, and alpha-5 variants. These targets also show high median pX values and high candidate active prevalence across the retained thresholds.

This indicates that the cleaned artifact is GABA-A-family focused but not uniformly distributed across receptor complex annotations. Downstream modeling and aggregation should account for this target-composition bias, since model performance may partly reflect the dominant high-potency target complexes represented in the training data.

In [34]:
# using our target_summary, create a target coverage table
target_coverage = target_summary.sort_values("n_records", ascending = False).copy()

target_coverage["pct_records"] = target_coverage["n_records"] / target_coverage["n_records"].sum()
target_coverage["cum_pct_records"] = target_coverage["pct_records"].cumsum()
target_coverage["pct_unique_molecules"] = target_coverage["n_unique_molecules"] / target_coverage["n_unique_molecules"].sum()
target_coverage["cum_pct_unique_molecules"] = target_coverage["pct_unique_molecules"].cumsum()

coverage_cols = [
    "target_chembl_id",
    "target_pref_name",
    "n_records",
    "pct_records",
    "cum_pct_records",
    "n_unique_molecules",
    "pct_unique_molecules",
    "cum_pct_unique_molecules"
]

target_coverage[coverage_cols].head(10)

,target_chembl_id,target_pref_name,n_records,pct_records,cum_pct_records,n_unique_molecules,pct_unique_molecules,cum_pct_unique_molecules
3,CHEMBL2094121,GABA-A receptor; alpha-1/beta-3/gamma-2,801,0.180609,0.180609,609,0.176266,0.176266
4,CHEMBL2094122,GABA-A receptor; alpha-5/beta-3/gamma-2,790,0.178129,0.358737,631,0.182634,0.358900
2,CHEMBL2094120,GABA-A receptor; alpha-3/beta-3/gamma-2,774,0.174521,0.533258,626,0.181187,0.540087
5,CHEMBL2094130,GABA-A receptor; alpha-2/beta-3/gamma-2,657,0.148140,0.681398,510,0.147612,0.687699
1,CHEMBL2093872,GABA-A receptor; anion channel,559,0.126043,0.807441,455,0.131693,0.819392
6,CHEMBL2095172,GABA-A receptor; alpha-1/beta-2/gamma-2,329,0.074183,0.881623,223,0.064544,0.883936
7,CHEMBL2095190,GABA-A receptor; alpha-6/beta-3/gamma-2,183,0.041263,0.922886,119,0.034443,0.918379
13,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,98,0.022097,0.944983,86,0.024891,0.943271
9,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,62,0.013980,0.958963,48,0.013893,0.957164
10,CHEMBL2111366,GABA A receptor alpha-4/beta-3/gamma-2,57,0.012852,0.971815,53,0.015340,0.972504


### Target Coverage Concentration

Target coverage is concentrated in a small number of GABA-A receptor complex annotations. The top target accounts for approximately 18% of cleaned activity evidence records, the top three targets account for approximately 53%, and the top five account for approximately 81%.

The highest-coverage targets are primarily alpha/beta-3/gamma-2 receptor complexes, indicating that the cleaned evidence artifact is GABA-A-family focused but not uniformly distributed across target annotations. This concentration should be considered during downstream aggregation, split design, and interpretation of model performance.

In [38]:
# tag the subset of targets (alpha/beta-3/gamma-2) to use later
df_clean["beta_3_gamma_2_tgt"] = (
    df_clean["target_pref_name"].str.contains("beta-3", case=False, na=False)
    & df_clean["target_pref_name"].str.contains("gamma-2", case=False, na=False)
)

# make (another) summary table including this subset to get more details about it
beta3_gamma2_summary = (
    df_clean
    .groupby("beta_3_gamma_2_tgt")
    .agg(
        n_records=("activity_id", "count"),
        n_unique_molecules=("molecule_chembl_id", "nunique"),
        mean_px=("px_value", "mean"),
        median_px=("px_value", "median"),
        pct_active_px_5_5=("active_px_5_5", "mean"),
        pct_active_px_6_0=("active_px_6_0", "mean"),
        pct_active_px_6_5=("active_px_6_5", "mean"),
    )
    .reset_index()
)

beta3_gamma2_summary

,beta_3_gamma_2_tgt,n_records,n_unique_molecules,mean_px,median_px,pct_active_px_5_5,pct_active_px_6_0,pct_active_px_6_5
0,False,1173,741,6.100741,6.270001,0.690537,0.572890,0.437340
1,True,3262,881,7.557981,7.638272,0.950644,0.913857,0.834764


### Beta-3/Gamma-2 Target Subset Audit

Because the top target-coverage table showed strong representation of beta-3/gamma-2-containing receptor complexes, a boolean target-subset flag was created to compare these records against the rest of the cleaned evidence set.

The beta-3/gamma-2 subset accounts for 3,262 of 4,435 cleaned records and 881 unique molecules. This subset also shows substantially higher observed potency than the remaining targets, with a median pX of approximately 7.64 compared with approximately 6.27 outside the subset. Candidate active prevalence is also much higher in the beta-3/gamma-2 subset; at `pX >= 6.0`, approximately 91.4% of beta-3/gamma-2 records are active compared with approximately 57.3% of non-beta-3/gamma-2 records.

This indicates that target-subset composition materially influences the cleaned evidence artifact. The beta-3/gamma-2 subset is a strong candidate for downstream subgroup analysis or possible scope refinement, but no target restriction is applied at this stage.

### **Section 7: Audit Repeated Molecule-Level Evidence**

The cleaned activity evidence dataset is still evidence-level, meaning a single molecule may appear in multiple records across endpoints, targets, assays, or publications. Before creating a molecule-level modeling table, we need to understand how repeated measurements are distributed and whether candidate activity labels conflict within the same molecule.

This audit will quantify records per molecule, endpoint and target coverage per molecule, assay repetition, and potential active/inactive disagreement under the retained candidate pX thresholds. These findings will inform the downstream aggregation policy used to collapse cleaned evidence records into one modeling row per molecule.

In [40]:
# create a molecule-level summary with one row per molecule_chembl_id
molecule_rep = (
    df_clean
    .groupby(["molecule_chembl_id", "canonical_smiles"])
    .agg(
        n_records = ("activity_id", "count"),
        n_endpoints = ("standard_type", "nunique"),
        n_targets = ("target_chembl_id", "nunique"),
        n_assays = ("assay_chembl_id", "nunique"),
        mean_px = ("px_value", "mean"),
        median_px = ("px_value", "median"),
        min_px = ("px_value", "min"),
        max_px = ("px_value", "max"),
    )
    .reset_index()
    .sort_values("n_records", ascending = False)
)

molecule_rep.head(10)

,molecule_chembl_id,canonical_smiles,n_records,n_endpoints,n_targets,n_assays,mean_px,median_px,min_px,max_px
97,CHEMBL12,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,50,3,9,50,7.574179,7.853872,0.908000,8.376751
295,CHEMBL17468,CCc1c(C(=O)OC)[nH]cc2nc3cc(OC)c(OC)cc3c1-2,38,2,6,38,8.168086,8.102930,6.872895,9.000000
1538,CHEMBL96,NCCCC(=O)O,32,3,12,24,5.711739,5.786954,4.096910,7.744727
665,CHEMBL286594,C#Cc1ccc2c(c1)C(=O)N(C)Cc1c(C(=O)OCC)ncn1-2,30,1,10,30,7.875699,7.588380,7.275724,9.309804
1514,CHEMBL911,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,28,3,7,28,6.833066,6.795880,5.732828,7.573489
1237,CHEMBL52030,CCOC(=O)c1ncn2c1[C@@H]1CCCN1C(=O)c1cc(OC)ccc1-2,27,1,5,27,7.862571,7.562249,7.079877,9.397940
1386,CHEMBL58757,C#Cc1ccc2c(c1)C(=O)N(C)Cc1c(C(=O)OC(C)(C)C)ncn1-2,27,2,10,27,7.367499,7.580044,3.488919,9.397940
1409,CHEMBL6597,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(N=[N+]=[N-])ccc1-2,22,1,9,22,8.546320,8.481486,7.481486,9.568636
213,CHEMBL13662,Cc1nnc2ccc(-c3cccc(C(F)(F)F)c3)nn12,20,1,4,20,6.312975,6.196852,5.698970,7.244125
727,CHEMBL300840,CN1Cc2c(C(=O)OC(C)(C)C)ncn2-c2ccc(C#C[Si](C)(C...,19,1,10,19,7.198357,6.844664,6.593460,8.583359


### Molecule-Level Repetition Audit

The cleaned evidence dataset contains repeated measurements for many molecules. Several compounds appear across many activity records, endpoints, targets, and assays. For example, the most repeated molecule has 50 cleaned records across three endpoint types and nine targets.

This confirms that cleaned activity evidence rows should not be used directly as independent model rows. A molecule-level aggregation policy is required before feature engineering and model training so that heavily studied compounds do not disproportionately influence downstream learning.

In [42]:
# now summarize n_records across all molecules
molecule_rep["n_records"].describe(
    percentiles = [0.5, 0.75, 0.9, 0.95, 0.99]
)

count    1546.000000
mean        2.868693
std         3.404389
min         1.000000
50%         2.000000
75%         4.000000
90%         6.000000
95%         8.000000
99%        15.000000
max        50.000000
Name: n_records, dtype: float64

In [43]:
# summarize this as well, need to know if repeated evidence is widespread or concentrated
molecule_rep["n_records"].value_counts().sort_index().head(20)

n_records
1     724
2     236
3     139
4     253
5      31
6      46
7      11
8      36
9       4
10     26
11      2
12      3
13      3
14     11
15      7
16      2
18      1
19      2
20      1
22      1
Name: count, dtype: int64

### Molecule-Level Repetition Summary

The cleaned activity evidence dataset contains 4,435 records across 1,546 unique molecules, confirming that many molecules have repeated evidence records. The median molecule has two records, while the mean is approximately 2.87 records per molecule.

The repetition summary shows that 724 molecules have only one cleaned evidence record, representing approximately 46.8% of the molecule set. The remaining 822 molecules have repeated measurements across one or more endpoints, targets, assays, or publications. Most repetition is modest, but a high-repeat tail exists: the 95th percentile molecule has eight records, the 99th percentile has fifteen records, and the most repeated molecule has fifty records.

This confirms that the modeling table should not be built directly from cleaned evidence rows. A molecule-level aggregation policy is required so that heavily studied compounds do not receive disproportionate influence during downstream feature engineering and model training.

In [44]:
# create a pX range column in molecule_rep
molecule_rep["px_range"] = molecule_rep["max_px"] - molecule_rep["min_px"]

# reinspect molecule_rep df with the new col px_range
molecule_rep[
    [
            "molecule_chembl_id",
            "n_records",
            "n_endpoints",
            "n_targets",
            "n_assays",
            "median_px",
            "min_px",
            "max_px",
            "px_range"
    ]
].sort_values("px_range", ascending = False).head(10)

,molecule_chembl_id,n_records,n_endpoints,n_targets,n_assays,median_px,min_px,max_px,px_range
1177,CHEMBL452,2,2,1,2,4.662791,0.255000,9.070581,8.815581
199,CHEMBL13280,16,3,8,16,8.602060,0.580000,8.958607,8.378607
631,CHEMBL268254,2,1,1,2,4.403921,0.255000,8.552842,8.297842
1405,CHEMBL63354,4,3,2,4,6.832773,0.550000,8.221849,7.671849
97,CHEMBL12,50,3,9,50,7.853872,0.908000,8.376751,7.468751
1541,CHEMBL9689,2,1,1,2,4.447451,0.740000,8.154902,7.414902
643,CHEMBL276821,2,1,1,2,4.552319,0.845000,8.259637,7.414637
1386,CHEMBL58757,27,2,10,27,7.580044,3.488919,9.397940,5.909021
212,CHEMBL13458,8,2,8,8,6.542121,2.544000,6.853872,4.309872
636,CHEMBL273481,18,3,5,12,6.529996,4.568636,8.698970,4.130334


### Molecule-Level pX Variability

Molecule-level pX ranges were computed to assess whether repeated evidence records are internally consistent. Several molecules show large potency ranges across their cleaned evidence records, with the largest observed pX range exceeding 8.8 units. Because a one-unit pX difference corresponds to a ten-fold concentration difference, these high-range cases represent substantial evidence variability rather than minor measurement noise.

High pX variability appears in both heavily studied molecules and molecules with only a small number of records. This indicates that repeated evidence may reflect different endpoint, target, assay, or context-specific activity rather than simple duplicate measurement. As a result, downstream molecule-level aggregation should avoid arbitrary row selection or optimistic max-only aggregation. A median pX policy remains a defensible starting point, but variability diagnostics such as `px_range`, `n_records`, `n_endpoints`, `n_targets`, and `n_assays` should be retained for interpretation and sensitivity analysis.

In [45]:
# next run the same percentile distribution that we just did prior for n_records
molecule_rep["px_range"].describe(
    percentiles = [0.5, 0.75, 0.90, 0.95, 0.99]
)

count    1546.000000
mean        0.574658
std         0.915479
min         0.000000
50%         0.002437
75%         0.995206
90%         1.689562
95%         2.036955
99%         3.490860
max         8.815581
Name: px_range, dtype: float64

In [47]:
# this time make a count table for common thresholds (run above first to observe the thresholds to use)
px_range_thresholds = [0, 0.5, 1.0, 2.0, 3.0]

px_range_rows = []

for threshold in px_range_thresholds:
    n_molecules = molecule_rep["px_range"].gt(threshold).sum()
    pct_molecules = molecule_rep["px_range"].gt(threshold).mean()

    px_range_rows.append({
        "px_range_threshold" : threshold,
        "n_molecules_above_threshold" : n_molecules,
        "pct_molecules_above_threshold" : pct_molecules
    })

pct_range_summary = pd.DataFrame(px_range_rows)

pct_range_summary

,px_range_threshold,n_molecules_above_threshold,pct_molecules_above_threshold
0,0.0,799,0.516818
1,0.5,571,0.369340
2,1.0,382,0.247089
3,2.0,85,0.054981
4,3.0,26,0.016818


### pX Variability Frequency Summary

Molecule-level pX ranges were summarized to quantify how common repeated-evidence variability is across the cleaned molecule set. Among 1,546 molecules, 799 molecules show at least some pX variation across repeated records, representing approximately 51.7% of the molecule set.

Moderate variability is common: 571 molecules have a pX range greater than 0.5, and 382 molecules have a pX range greater than 1.0. Because a one-unit pX difference corresponds to a ten-fold concentration difference, approximately 24.7% of molecules show at least a ten-fold potency spread across cleaned evidence records.

Severe variability is less common but still present. Eighty-five molecules have a pX range greater than 2.0, and twenty-six molecules have a pX range greater than 3.0. These high-variability cases support retaining conflict diagnostics during molecule-level aggregation and avoiding optimistic max-only aggregation as the primary label policy.

In [49]:
# make new summary for rows that have label conflicts

label_conflict_rows = []

for label_col in candidate_label_cols:
    molecule_label_audit = (
        df_clean
        .groupby("molecule_chembl_id")
        .agg(
            min_label = (label_col, "min"),
            max_label = (label_col, "max"),
            mean_label = (label_col, "mean"),
            n_unique_labels = (label_col, "nunique"),
            n_records = ("activity_id", "count")
        )
        .reset_index()
    )

    label_conflict = molecule_label_audit["n_unique_labels"].gt(1)

    label_conflict_rows.append({
        "label_col" : label_col,
        "n_molecules" : len(molecule_label_audit),
        "n_conflict_molecules" : label_conflict.sum(),
        "pct_conflict_molecules" : label_conflict.mean()
    })

label_conflict_summary = pd.DataFrame(label_conflict_rows)

label_conflict_summary



,label_col,n_molecules,n_conflict_molecules,pct_conflict_molecules
0,active_px_5_5,1546,56,0.036223
1,active_px_6_0,1546,101,0.065330
2,active_px_6_5,1546,183,0.118370


### Candidate Label Conflict Summary

Candidate activity labels were audited for within-molecule disagreement across repeated evidence records. A molecule was considered to have a label conflict if its cleaned evidence records contained both active and inactive labels under a given pX threshold.

Conflict rates were relatively low overall but increased as the activity threshold became stricter. At `pX >= 5.5`, 56 of 1,546 molecules showed conflicting labels, representing approximately 3.6% of the molecule set. At `pX >= 6.0`, 101 molecules showed conflicting labels, or approximately 6.5%. At `pX >= 6.5`, conflicts increased to 183 molecules, or approximately 11.8%.

These results indicate that repeated evidence usually agrees under the candidate label definitions, but threshold choice affects label stability. The `pX >= 6.0` threshold remains a defensible leading candidate because it improves inactive-class contrast while keeping within-molecule label conflict relatively limited. The `pX >= 5.5` and `pX >= 6.5` thresholds should remain available for downstream sensitivity analysis.

In [55]:
# inspect further into active_px_6_0, display the top conflict molecules sorted by n_records
# use the template from our last summary

label_col = "active_px_6_0"

molecule_label_audit = (
    df_clean
    .groupby("molecule_chembl_id")
    .agg(
        min_label = (label_col, "min"),
        max_label = (label_col, "max"),
        mean_label = (label_col, "mean"),
        n_unique_labels = (label_col, "nunique"),
        n_records = ("activity_id", "count"),
        n_endpoints=("standard_type", "nunique"),
        n_targets=("target_chembl_id", "nunique"),
        n_assays=("assay_chembl_id", "nunique"),
        median_px=("px_value", "median"),
        min_px=("px_value", "min"),
        max_px=("px_value", "max"),
    )
    .reset_index()
)

molecule_label_audit["px_range"] = (
    molecule_label_audit["max_px"] - molecule_label_audit["min_px"]
)

molecule_label_audit["has_label_conflict"] = (
    molecule_label_audit["n_unique_labels"].gt(1)
)

molecule_label_audit.head(10)

# then for better readability and filtered cols
conflict_molecules_6_0 = (
    molecule_label_audit
    .loc[molecule_label_audit["has_label_conflict"]]
    .sort_values("px_range", ascending=False)
)

conflict_molecules_6_0[
    [
        "molecule_chembl_id",
        "has_label_conflict",
        "n_records",
        "n_endpoints",
        "n_targets",
        "n_assays",
        "median_px",
        "min_px",
        "max_px",
        "px_range",
        "mean_label",
    ]
].head(10)


,molecule_chembl_id,has_label_conflict,n_records,n_endpoints,n_targets,n_assays,median_px,min_px,max_px,px_range,mean_label
1177,CHEMBL452,True,2,2,1,2,4.662791,0.255000,9.070581,8.815581,0.500000
199,CHEMBL13280,True,16,3,8,16,8.602060,0.580000,8.958607,8.378607,0.937500
631,CHEMBL268254,True,2,1,1,2,4.403921,0.255000,8.552842,8.297842,0.500000
1405,CHEMBL63354,True,4,3,2,4,6.832773,0.550000,8.221849,7.671849,0.500000
97,CHEMBL12,True,50,3,9,50,7.853872,0.908000,8.376751,7.468751,0.960000
1541,CHEMBL9689,True,2,1,1,2,4.447451,0.740000,8.154902,7.414902,0.500000
643,CHEMBL276821,True,2,1,1,2,4.552319,0.845000,8.259637,7.414637,0.500000
1386,CHEMBL58757,True,27,2,10,27,7.580044,3.488919,9.397940,5.909021,0.814815
212,CHEMBL13458,True,8,2,8,8,6.542121,2.544000,6.853872,4.309872,0.875000
636,CHEMBL273481,True,18,3,5,12,6.529996,4.568636,8.698970,4.130334,0.944444


### Inspection of `pX >= 6.0` Conflict Molecules

Conflict molecules under the leading `pX >= 6.0` candidate label were inspected to determine whether label disagreement was driven by sparse evidence or by broader endpoint, target, and assay diversity.

The highest-variability conflict molecules include both sparse two-record cases and heavily studied multi-target compounds. Several two-record molecules have `mean_label = 0.5`, indicating one active and one inactive evidence record. These cases are fragile because the final molecule-level label could change depending on the aggregation policy.

Other conflict molecules have many records and high `mean_label` values, indicating that most evidence records are active despite at least one inactive record. These cases are less ambiguous than the sparse 50/50 conflicts but still show that repeated evidence can span a wide pX range. Multi-endpoint and multi-target conflict molecules suggest that some disagreement may reflect biological or assay-context differences rather than simple duplicate measurement noise.

In [60]:
# finalize this section with a last summary table, using our previous conflict_molecules_6_0 table

conflict_profile_summary = pd.DataFrame([
    {
    "n_conflict_molecules" : len(conflict_molecules_6_0),

    "n_two_record_conflicts" : conflict_molecules_6_0["n_records"].eq(2).sum(),
    "pct_two_record_conflicts" : conflict_molecules_6_0["n_records"].eq(2).mean(),

    "n_mostly_active_conflicts" : conflict_molecules_6_0["mean_label"].ge(0.8).sum(),
    "pct_mostly_active_conflicts" : conflict_molecules_6_0["mean_label"].ge(0.8).mean(),

    "n_high_variability_conflicts" : conflict_molecules_6_0["px_range"].gt(2).sum(),
    "pct_high_variability_conflicts" : conflict_molecules_6_0["px_range"].gt(2).mean(),
    }
])

conflict_profile_summary

,n_conflict_molecules,n_two_record_conflicts,pct_two_record_conflicts,n_mostly_active_conflicts,pct_mostly_active_conflicts,n_high_variability_conflicts,pct_high_variability_conflicts
0,101,17,0.168317,22,0.217822,48,0.475248


### Conflict Profile at `pX >= 6.0`

The leading candidate label threshold, `pX >= 6.0`, produced 101 molecule-level label conflicts. These conflict molecules were profiled to determine whether disagreement was primarily driven by sparse evidence, mostly-active evidence with occasional inactive records, or broader pX variability.

Among the 101 conflict molecules, 17 had exactly two evidence records, representing approximately 16.8% of the conflict set. Twenty-two conflict molecules were mostly active, with at least 80% of their evidence records labeled active. Forty-eight conflict molecules had a pX range greater than 2.0, representing approximately 47.5% of the conflict set.

This indicates that label conflicts are not primarily limited to simple two-record edge cases. Many conflict molecules show substantial potency variability across their cleaned evidence records, supporting the use of a robust molecule-level aggregation policy such as median pX. These molecules should not be removed by default, but conflict diagnostics should be retained for downstream interpretation and sensitivity analysis.

### Molecule-Level Aggregation Policy Recommendation

The repeated-evidence audit shows that cleaned activity records should be collapsed to molecule-level rows before feature engineering and modeling. More than half of molecules have repeated evidence, and some molecules show substantial variation across endpoints, targets, and assays. Simple row deduplication would be inappropriate because repeated records often represent distinct biological or assay evidence rather than duplicate observations.

For BioPred v1, the recommended primary aggregation policy is to compute one molecule-level row per `molecule_chembl_id` using the median pX value across retained cleaned evidence records. The median provides a robust central potency estimate while reducing the influence of isolated extreme records. This is more defensible than randomly retaining one evidence row or using an optimistic max-pX policy.

The leading candidate molecule-level activity label should be derived after aggregation using:

`active_median_px_6_0 = 1 if median_px >= 6.0 else 0`

The `pX >= 6.0` threshold remains the leading candidate because it provides better inactive-class contrast than `pX >= 5.5` while maintaining relatively limited within-molecule label conflict compared with `pX >= 6.5`. The `pX >= 5.5` and `pX >= 6.5` thresholds should be retained as sensitivity-analysis candidates rather than discarded.

The molecule-level modeling table should also preserve diagnostic fields from the evidence audit, including record count, endpoint count, target count, assay count, minimum pX, maximum pX, pX range, and candidate-label conflict indicators. These diagnostics should not be used as default exclusion filters. Instead, they should support interpretation, model-card limitations, and optional sensitivity checks.

The recommended BioPred v1 policy is therefore:

- Primary potency aggregation: median pX per molecule.
- Primary label candidate: median pX >= 6.0.
- Sensitivity label candidates: median pX >= 5.5 and median pX >= 6.5.
- Retain conflict and variability diagnostics.
- Do not remove high-conflict or high-variability molecules by default.
- Do not use max pX as the primary aggregation policy.

### **Section 8: Save Cleaned Activity Evidence Artifact**

This notebook produces a cleaned activity evidence artifact for downstream molecule-level aggregation, feature engineering, and split planning. The saved artifact remains evidence-level: each row represents one cleaned ChEMBL activity evidence record, not one final modeling molecule.

The artifact includes standardized pX values, candidate evidence-level activity labels, target-subset indicators, and repeated-evidence diagnostics needed for downstream aggregation. Final molecule-level labels are not selected in this notebook. Instead, this notebook preserves the information required to construct and compare molecule-level aggregation policies in the next stage.

Before saving, the artifact should be checked for expected schema, row count, molecule count, missing values in required fields, and candidate-label columns. The saved file should represent the reproducible handoff from evidence cleaning to molecule-level aggregation.

In [63]:
# re-examine and check our df artifact before saving in df_clean
print(df_clean.shape)
print(df_clean.columns)
print(df_clean["molecule_chembl_id"].nunique())
print(df_clean.isna().sum())


(4435, 27)
Index(['activity_id', 'target_chembl_id', 'target_pref_name', 'target_type',
       'organism', 'assay_id', 'assay_chembl_id', 'assay_type',
       'confidence_score', 'molregno', 'molecule_chembl_id',
       'canonical_smiles', 'standard_inchi_key', 'molecule_type', 'max_phase',
       'standard_type', 'standard_relation', 'standard_value',
       'standard_units', 'pchembl_value', 'doc_id', 'standard_value_molar',
       'px_value', 'active_px_5_5', 'active_px_6_0', 'active_px_6_5',
       'beta_3_gamma_2_tgt'],
      dtype='object')
1546
activity_id                0
target_chembl_id           0
target_pref_name           0
target_type                0
organism                   0
assay_id                   0
assay_chembl_id            0
assay_type                 0
confidence_score           0
molregno                   0
molecule_chembl_id         0
canonical_smiles           0
standard_inchi_key         0
molecule_type              0
max_phase               4138
standar

### Cleaned Evidence Artifact Validation

Before saving the cleaned activity evidence artifact, we validated the final dataframe shape, molecule count, schema, and missingness profile. The final `df_clean` artifact contains 4,435 cleaned evidence records across 1,546 unique molecules.

The artifact includes activity identifiers, molecule identifiers, canonical SMILES, target annotations, assay identifiers, standardized pX values, candidate evidence-level activity labels, and the beta-3/gamma-2 target subset flag. Required handoff fields contain no missing values, supporting use of this artifact in the next notebook for molecule-level aggregation and feature generation.

In [64]:
# save our df_clean to the PROCESSED_DIR for use in our next notebook
clean_activity_evidence_path = (
    paths.PROCESSED_DATA_DIR / "chembl_gabaa_clean_activity_evidence.parquet"
)

df_clean.to_parquet(clean_activity_evidence_path, index=False)

# check to see it was sent
print(clean_activity_evidence_path)
print(df_clean.shape)

/home/ryanm/projects/BioPred/data/processed/chembl_gabaa_clean_activity_evidence.parquet
(4435, 27)


### Notebook 3 Closeout

This notebook converted the provisional GABA-A ChEMBL activity evidence extract into a validated cleaned activity evidence artifact. The workflow audited endpoint composition, units, relation operators, molecule types, assay types, target annotations, and confidence scores before applying a conservative usability filter.

The final cleaned evidence artifact contains 4,435 exact nanomolar small-molecule activity records across 1,546 unique molecules. Standardized full-precision pX values were computed from `standard_value`, validated against ChEMBL `pchembl_value`, and used to create candidate evidence-level activity labels at `pX >= 5.5`, `pX >= 6.0`, and `pX >= 6.5`.

Target composition and beta-3/gamma-2 subset audits showed that the cleaned evidence set is GABA-A-family focused but not uniformly distributed across receptor complex annotations. Repeated-evidence analysis showed that many molecules have multiple activity records across endpoints, targets, and assays, confirming that evidence rows should not be used directly as independent model rows.

The recommended downstream aggregation policy is to create one molecule-level row per `molecule_chembl_id` using median pX as the primary potency summary. The leading candidate molecule-level label is `median_px >= 6.0`, with `median_px >= 5.5` and `median_px >= 6.5` retained for sensitivity analysis. Conflict and variability diagnostics should be preserved for interpretation but should not be used as default exclusion filters.

The cleaned evidence artifact was saved to the processed data directory as the handoff to the next notebook. The next stage will aggregate evidence to molecule-level rows, create molecule-level labels, generate molecular features, and prepare split-ready modeling datasets.
